# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Station

In [12]:
snap = station.snapshot(update=True) # <- updates parameters in station

In [13]:
verticalsnap = snap['instruments']['mso5']['submodules']['channels']['channels']['mso5_ch1']['parameters'].keys()

In [14]:
horizontalsnap = snap['instruments']['mso5']['parameters'].keys()

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import quick_check
from functions import calibrate
import snspd
# params = snspd.snspd(config) # <- add this later 
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260413-22476-qcodes.log
Experiment loaded. Last ID no: 464


In [2]:
station = Station(config_file="friesland.yaml")

In [11]:
dmm = station.load_instrument("dmm", revive_instance=True)

Connected to: Agilent Technologies 34410A (serial:MY47027892, firmware:2.35-2.35-0.09-46-09) in 0.08s


In [12]:
yoko = station.load_instrument("yoko", revive_instance=True)

Connected to: YOKOGAWA GS210 (serial:91T928105, firmware:2.02) in 0.03s


In [3]:
laser = station.load_instrument("laser", revive_instance=True)

2026-04-13 15:31:49,652 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ D:\SNSPD\SNSPD2\PPCL550.py:4: QCoDeSDeprecationWarning: The `qcodes.utils.helpers` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.utils.helpers import create_on_off_val_mapping



Connected to: PurePhotonic PPCL550 (serial:PP70AJ005, firmware:PV 2.0.0:HW 3.0.0:FW 7.0.0:AS C1:OT 1.0.0) in 1.63s


In [14]:
MS = station.load_instrument("mso5", revive_instance=True)

2026-04-09 17:11:35,797 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ D:\SNSPD\SNSPD2\MSO5.py:7: QCoDeSDeprecationWarning: The `qcodes.instrument.base` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.instrument.base import InstrumentBase



In [18]:
pm100usb = station.load_instrument("pm100usb", revive_instance=True)

In [19]:
pms120 = station.load_instrument("pms120", revive_instance=True)

In [17]:
tc = station.load_instrument("fridge", revive_instance=True)

In [20]:
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

In [5]:
laser.power()

7.0

# Calibration

Finding voltage to be applied to VA1550 to give ~100dB attenuation

In [26]:
ID = 457 
data = load_by_id(ID).get_parameter_data()
idx = 26 # ID corresponding to 4.7V attenuation, ~99.2dB total
print(data['v_attenuator']['v_attenuator'][idx])
print(data['total_attenuation']['total_attenuation'][idx])

4.7
99.1683535121345
